# Colab: Hugging Face Tokenizers (explained deeply)

Lives in **`7_HUGGING_FACE/`**. Upload / open in **Google Colab** to run.

**How this notebook is organized:** **code cell first**, then **explanation** under it.

Related:
- [`colab_pipelines_explained.ipynb`](./colab_pipelines_explained.ipynb) — pipeline uses a tokenizer **inside**
- [`../3_LLMs_and_TOKENS/what_are_tokens.md`](../3_LLMs_and_TOKENS/what_are_tokens.md) — token ideas

Optional: Colab Secret `HF_TOKEN` if downloads need auth.

Run **top → bottom**.

## Big picture — why tokenizers?

Models do **not** read English letters. They read **numbers** (token ids).

```text
"I love pipelines!"
        ↓  tokenizer
  [101, 1045, 2293, ...]   ← token ids
        ↓  model
     scores / next tokens
        ↓  tokenizer decode
   human-readable text again
```

| Piece | Job |
|-------|-----|
| **Tokenizer** | Text ↔ token ids |
| **Model** | Math on those ids |
| **Pipeline** | Calls tokenizer + model for you |

Different models often have **different** tokenizers (different vocab / splits).

## Setup

In [ ]:
!pip -q install transformers

### Explanation — install

`transformers` includes `AutoTokenizer` (and models/pipelines). Quiet install with `-q`.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("HF_TOKEN: logged in")
    else:
        print("HF_TOKEN: not set (OK for many public tokenizers)")
except Exception as e:
    print("Secrets / login skipped:", e)

### Explanation — HF token (needed for Llama 3)

Many public tokenizers (DistilBERT, GPT-2) work without login.

**Llama 3 is gated** — Example 1 needs:

1. `HF_TOKEN` in Colab Secrets (or local `.env`)
2. Accept access on the Llama model page on Hugging Face
3. `login(token=...)` succeeds in the Setup cell above


## Example 1 — Llama 3 tokenizer (course-style first example)

This is the first tokenizer example from the course vibe: load **Llama 3**’s tokenizer from the Hub and see token ids + pieces.

**Needs:** Colab Secret / env `HF_TOKEN`, and your Hugging Face account must have **accepted Llama access** on the model page (gated model).


In [ ]:
from transformers import AutoTokenizer

# Llama 3 Instruct tokenizer (gated on the Hub — needs HF login + access approved)
llama_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer_llama = AutoTokenizer.from_pretrained(llama_id)

text = "I am excited to learn about LLM tokenizers!"

token_ids = tokenizer_llama.encode(text)
token_pieces = tokenizer_llama.convert_ids_to_tokens(token_ids)
decoded = tokenizer_llama.decode(token_ids)

print("model:", llama_id)
print("token ids:", token_ids)
print("token pieces:", token_pieces)
print("decoded:", decoded)
print("number of tokens:", len(token_ids))


### Explanation — Llama 3 first example

#### What this cell does

```text
text  →  Llama 3 tokenizer  →  token ids + pieces  →  decode back
```

| Line | Meaning |
|------|---------|
| `AutoTokenizer.from_pretrained(llama_id)` | Download **Llama 3’s** tokenizer (not DistilBERT, not GPT-2) |
| `encode(text)` | Text → list of **token ids** (integers) |
| `convert_ids_to_tokens(...)` | Show the **string chunks** for those ids |
| `decode(...)` | Ids → text again |
| `len(token_ids)` | How many tokens this sentence became for Llama |

#### Why you may see `<|begin_of_text|>`

Llama 3 often adds a **special token** at the start (beginning of text). That is normal — it is part of how the model expects prompts.

#### Why this can fail without setup

Llama models on Hugging Face are often **gated**:

1. Create / use `HF_TOKEN` (Colab Secret or `.env`)
2. Open the model page on huggingface.co and **request / accept access**
3. Log in (your Setup cell with `login(token=...)`)

If access is missing, `from_pretrained` errors — not because your code is wrong.

#### Why start with Llama (not DistilBERT)?

The course starts with a **real chat LLM tokenizer** so you see modern special tokens and splits.  
Later examples in this notebook use smaller public models (DistilBERT / GPT-2) that work even without gated access.

#### Optional: same idea with a public Llama-family-style model

If gated Llama access is blocked, you can temporarily try a public instruct model id later — but for the **course first example**, keep Llama 3 as above once access is approved.


## Example 2 — Load a tokenizer from the Hub (DistilBERT)

Same DistilBERT family you saw in the pipelines notebook (easy, small).


In [ ]:
from transformers import AutoTokenizer

model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(type(tokenizer))
print("vocab size:", tokenizer.vocab_size)

### Explanation — `AutoTokenizer`

```python
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
```

| Piece | Meaning |
|-------|---------|
| `AutoTokenizer` | Helper that picks the **right tokenizer class** for that Hub model |
| `from_pretrained(model_id)` | Download tokenizer files that **match** that model |
| `tokenizer` | Object that can encode / decode text |
| `vocab_size` | How many token ids this tokenizer knows |

**Rule:** always use the tokenizer that belongs to the **same** model id (or family) you will run.

```text
Wrong:  Model A + Tokenizer B
Right:  Model A + Tokenizer A
```

## Example 3 — Text → token ids → text again


In [ ]:
text = "I love learning Hugging Face pipelines!"

encoded = tokenizer(text)
print("encoded keys:", encoded.keys())
print("input_ids:", encoded["input_ids"])

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("tokens:", tokens)

decoded = tokenizer.decode(encoded["input_ids"])
print("decoded:", decoded)

### Explanation — encode / see pieces / decode

| Line | Meaning |
|------|---------|
| `tokenizer(text)` | Encode: text → numbers (+ attention mask, etc.) |
| `input_ids` | The token id list the model would see |
| `convert_ids_to_tokens(...)` | Show the **string pieces** for those ids |
| `decode(...)` | Numbers → readable text again |

You may see special tokens like `[CLS]` / `[SEP]` (BERT-style). Those mark start/end for the model.

```text
text → input_ids → (model would run here) → decode → text
```

In the pipelines notebook, `pipeline(...)` does this encoding step **for you**.

## Example 4 — Same sentence, two different tokenizers

Proof that **tokenizers are not universal**.


In [ ]:
text = "Hugging Face tokenizers split text into chunks."

tok_a = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)
tok_b = AutoTokenizer.from_pretrained("gpt2")

ids_a = tok_a.encode(text)
ids_b = tok_b.encode(text)

print("DistilBERT ids:", ids_a)
print("DistilBERT pieces:", tok_a.convert_ids_to_tokens(ids_a))
print()
print("GPT-2 ids:", ids_b)
print("GPT-2 pieces:", tok_b.convert_ids_to_tokens(ids_b))
print()
print("same number of tokens?", len(ids_a) == len(ids_b))

### Explanation — why splits differ (Example 4)

#### What this cell proves

You use the **same English sentence**, but two Hub tokenizers:

| Variable | Tokenizer | From model id |
|----------|-----------|----------------|
| `tok_a` | DistilBERT-style | `distilbert/...-sst-2-english` |
| `tok_b` | GPT-2 | `gpt2` |

Then:

```python
ids_a = tok_a.encode(text)
ids_b = tok_b.encode(text)
```

`encode(text)` → list of **token ids** (integers).

`convert_ids_to_tokens(...)` → the **string chunks** those ids stand for (so you can *see* the split).

#### Line by line

| Line | Meaning |
|------|---------|
| `text = "..."` | One shared sentence for a fair comparison |
| `AutoTokenizer.from_pretrained(...)` | Load each tokenizer’s vocab + rules from the Hub |
| `tok_a.encode(text)` | DistilBERT’s numbers for that sentence |
| `tok_b.encode(text)` | GPT-2’s numbers for the **same** sentence |
| `convert_ids_to_tokens(...)` | Show pieces like `hugging`, `##face`, or `ĠFace` |
| `len(ids_a) == len(ids_b)` | Often **False** — different tokenizers → different counts |

#### Why the outputs look different

| Why | Meaning |
|-----|---------|
| Different vocab | Each tokenizer was built with a different “dictionary” of chunks |
| Different algorithms | e.g. WordPiece (BERT family) vs BPE (GPT-2) |
| Special tokens | DistilBERT often adds `[CLS]` / `[SEP]`; GPT-2 usually does not |
| Case / spaces | DistilBERT here is **uncased**; GPT-2 keeps case; spaces may become weird symbols like `Ġ` |

So you might see:

```text
Same sentence
  DistilBERT →  [CLS] hugging face ... [SEP]   (more/fewer ids)
  GPT-2      →  Hugging ĠFace ...              (different ids)
```

**Takeaway:** token count and splits depend on the **tokenizer**, not only on the sentence.

#### Why this matters

- Billing / context limits are in **tokens** → same prompt can cost different amounts on different models  
- Never mix Model A with Tokenizer B  
- Your OpenAI practice used **tiktoken**; this notebook uses **HF AutoTokenizer** — same idea, different tool  

Try changing `text` to a rare word (e.g. an invented name) and watch both tokenizers break it into pieces.


## Example 5 — Batch / tensors (what models often need)


In [ ]:
texts = [
    "I love this!",
    "This movie was terrible.",
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

print(batch)
print("input_ids shape:", batch["input_ids"].shape)

### Explanation — padding, truncation, tensors (Example 5)

#### What is the “padding rectangle”? (read this first)

Each sentence becomes a **row of boxes**. One box = one token id.

Without padding, rows have **different lengths** (uneven / ragged):

```text
"Hi!"                 →  ■ ■ ■
"I love this movie"   →  ■ ■ ■ ■ ■ ■
```

Models like a **table**: every row must have the **same number of columns**.

**Padding** = fill short rows with empty boxes (`[PAD]`) until they match the longest row:

```text
Before (NOT a rectangle):

row 0:  ■ ■ ■
row 1:  ■ ■ ■ ■ ■ ■

After padding (NOW a rectangle):

row 0:  ■ ■ ■ □ □ □
row 1:  ■ ■ ■ ■ ■ ■

■ = real token
□ = PAD (filler only)
```

Now it is a clean **2 × 6 rectangle** (2 sentences, 6 columns).

```text
shape = (2, 6)
         │  └─ width  = tokens per row (same for all)
         └──── height = number of sentences
```

That is all “padding rectangle” means: **make uneven rows into a same-width grid.**

#### Why `attention_mask`?

The model must not treat `□` pads as real words.

```text
attention_mask:

row 0:  1 1 1 0 0 0   ← look at real tokens only
row 1:  1 1 1 1 1 1
```

`1` = real, `0` = pad (ignore).

#### What the code arguments do

| Argument | Meaning |
|----------|---------|
| `texts` | List of sentences (a batch) |
| `padding=True` | Build that rectangle with PAD fillers |
| `truncation=True` | If a sentence is too long for the model, cut it |
| `return_tensors="pt"` | Return a PyTorch tensor table, not Python lists |

#### Tiny story

```text
many sentences → different lengths → PAD short ones → rectangle → model
```

Try after running the cell:

```python
print(batch["input_ids"])
print(batch["attention_mask"])
```

You should see short rows padded, and zeros in the mask where pads are.


## Example 6 — Connect back to pipeline

Pipeline = tokenizer + model + postprocess (you already practiced this).


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

print(classifier("I love learning Hugging Face tokenizers!"))

### Explanation — pipeline still uses a tokenizer

```text
pipeline
   │
   ├── tokenizer   ← what this notebook teaches
   ├── pretrained model
   ├── preprocessing
   └── postprocessing
```

You pass a string → pipeline tokenizes → model runs → you get `POSITIVE` / `NEGATIVE`.

Now you know **what’s inside** the first step.

## Recap

1. Models read **token ids**, not raw English.
2. `AutoTokenizer.from_pretrained(model_id)` loads the matching Hub tokenizer.
3. Encode → numbers; decode → text; `convert_ids_to_tokens` → see pieces.
4. Different models → different tokenizers → different splits / counts.
5. `pipeline` hides the tokenizer; this notebook shows it.

```text
3_LLMs_and_TOKENS:  ideas + OpenAI tiktoken
7_HUGGING_FACE:     AutoTokenizer + pipelines (this notebook + pipelines notebook)
```

### Try next

- Change `text` and compare DistilBERT vs GPT-2 splits again
- Tokenize a rare / invented word and watch it break into pieces
- Re-open [`colab_pipelines_explained.ipynb`](./colab_pipelines_explained.ipynb) Example 3 (manual tokenizer + model)